# 소방 RC카 불 감지 모델 학습 (Google Colab)

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고른다.
2. 위에서부터 셀을 하나씩 실행한다 (▶ 또는 Shift+Enter).
3. 마지막 셀이 `fire.onnx`를 내려받는다. 그 파일을 Pi의 `/home/song/proj/fire/models/fire.onnx`에 넣는다.

데이터: D-Fire 일부 (CC0 1.0, github.com/gaiasd/DFireDataset), 불 클래스만. 자세한 건 프로젝트의 `docs/fire.md`.

In [ ]:
# 1) GPU 확인: 'Tesla T4' 같은 이름이 나와야 한다. 안 나오면 런타임 유형을 GPU로 바꾼다.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) 데이터셋 가져오기
# Google Drive '내 드라이브'에 dfire_fire.zip을 올려 두었으면 Drive에서 읽고 (빠름, 권장),
# 아니면 이 컴퓨터에서 직접 고르는 창이 뜬다 (220MB라 몇 분 걸린다).
import os, shutil
zip_path = '/content/dfire_fire.zip'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive/dfire_fire.zip'):
        shutil.copy('/content/drive/MyDrive/dfire_fire.zip', zip_path)
except Exception as error:
    print('Drive 사용 안 함:', error)
if not os.path.exists(zip_path):
    from google.colab import files
    uploaded = files.upload()
    shutil.move(next(iter(uploaded)), zip_path)
!rm -rf /content/dfire && mkdir -p /content/dfire && unzip -q {zip_path} -d /content/dfire
!ls /content/dfire/images/*/ | head -3; for s in train val test; do echo $s $(ls /content/dfire/images/$s | wc -l); done

In [ ]:
# 3) 학습 도구 설치
!pip install -q ultralytics onnx onnxslim

In [ ]:
# 4) 학습 (T4 GPU에서 약 15~30분). imgsz=320은 Pi의 fire/config.toml input_size와 같아야 한다.
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/dfire/data_colab.yaml', imgsz=320, epochs=60, batch=64, patience=15,
            device=0, workers=2, project='/content/runs', name='fire', exist_ok=True)

In [ ]:
# 5) 학습에 쓰지 않은 시험 사진(600장)으로 성능 확인. 이 출력 전체를 복사해 두면 좋다.
best = YOLO('/content/runs/fire/weights/best.pt')
metrics = best.val(data='/content/dfire/data_colab.yaml', split='test', imgsz=320, device=0)
print('precision', round(metrics.box.mp, 3), 'recall', round(metrics.box.mr, 3),
      'mAP50', round(metrics.box.map50, 3), 'mAP50-95', round(metrics.box.map, 3))

In [ ]:
# 6) Pi용 ONNX로 바꾸고 내려받기 (Drive에도 복사)
onnx_path = best.export(format='onnx', imgsz=320, opset=12, simplify=True, dynamic=False)
shutil.copy(onnx_path, '/content/fire.onnx')
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy('/content/fire.onnx', '/content/drive/MyDrive/fire.onnx')
    shutil.copy('/content/runs/fire/results.png', '/content/drive/MyDrive/fire_results.png')
    print('Drive에 fire.onnx 저장함')
from google.colab import files
files.download('/content/fire.onnx')